In [1]:

import pandas as pd
import joblib

from pathlib import Path
from scipy.sparse import load_npz
from sklearn.naive_bayes import MultinomialNB

# Path to the saved artifacts
MODEL_DIR = Path("../models")

# Load the TF-IDF training and testing matrices
X_train_tfidf = load_npz(MODEL_DIR / "X_train_tfidf.npz")
X_test_tfidf = load_npz(MODEL_DIR / "X_test_tfidf.npz")

# Load the corresponding labels
y_train = pd.read_csv(MODEL_DIR / "y_train.csv").squeeze("columns")
y_test = pd.read_csv(MODEL_DIR / "y_test.csv").squeeze("columns")

# Load the fitted TF-IDF vectorizer for future predictions
tfidf_vectorizer = joblib.load(
    MODEL_DIR / "tfidf_vectorizer.joblib"
)

# Verify the loaded artifacts
print("Training matrix:", X_train_tfidf.shape)
print("Testing matrix:", X_test_tfidf.shape)
print("Training labels:", y_train.shape)
print("Testing labels:", y_test.shape)
print("Vocabulary size:", len(tfidf_vectorizer.vocabulary_))


Training matrix: (4135, 7591)
Testing matrix: (1034, 7591)
Training labels: (4135,)
Testing labels: (1034,)
Vocabulary size: 7591


In [2]:

# Initialize the Multinomial Naive Bayes classifier
nb_model = MultinomialNB()

# Train the model using the training TF-IDF features and labels
nb_model.fit(X_train_tfidf, y_train)

print("Model training completed successfully!")


Model training completed successfully!


In [3]:

# Define the path for the trained model
model_path = MODEL_DIR / "multinomial_nb_model.joblib"

# Save the trained model
joblib.dump(nb_model, model_path)

print("Trained model saved to:", model_path)


Trained model saved to: ../models/multinomial_nb_model.joblib


In [6]:
import joblib
from pathlib import Path

PROJECT_DIR = Path.cwd().parent
MODEL_PATH = PROJECT_DIR / "models" / "multinomial_nb_model.joblib"

model = joblib.load(MODEL_PATH)

print("Trained model loaded successfully!")

Trained model loaded successfully!


In [7]:
from sklearn.metrics import accuracy_score, f1_score

y_train_pred = model.predict(X_train_tfidf)
y_test_pred = model.predict(X_test_tfidf)

print("Training Accuracy:", round(accuracy_score(y_train, y_train_pred), 4))
print("Testing Accuracy: ", round(accuracy_score(y_test, y_test_pred), 4))

print("Training F1-score:", round(f1_score(y_train, y_train_pred), 4))
print("Testing F1-score: ", round(f1_score(y_test, y_test_pred), 4))

Training Accuracy: 0.9686
Testing Accuracy:  0.9516
Training F1-score: 0.8578
Testing F1-score:  0.7642


## Model Training — Summary

### 1. Loading Artifacts
Loaded the TF-IDF feature matrices, corresponding labels, and fitted TF-IDF vectorizer created in Notebook 03.

### 2. Model Selection
Selected **Multinomial Naive Bayes** as the baseline classifier because it is commonly used for text-classification tasks and supports non-negative TF-IDF features.

### 3. Model Training
Trained the classifier using:
- Training TF-IDF matrix: 4,135 messages × 7,591 features
- Training labels: 4,135 corresponding ham/spam labels

The model learned patterns associated with legitimate and spam messages from the training data.

### 4. Model Saving
Saved the trained classifier to:

`models/multinomial_nb_model.joblib`

This allows the model to be loaded later without retraining.

### 5. Training and Testing Performance

Compared the model's performance on the training set and the held-out test set.

| Metric | Training | Testing |
|---|---:|---:|
| Accuracy | 96.86% | 95.16% |
| Spam F1-score | 85.78% | 76.42% |

The accuracy gap is 1.70 percentage points, while the spam F1-score gap is 9.36 percentage points.

These results suggest that overall accuracy generalizes reasonably well, although spam-classification performance declines on unseen data.

### Conclusion

The baseline Multinomial Naive Bayes classifier has been trained, saved, and evaluated. Although the small accuracy gap does not indicate severe overfitting by itself, the larger F1-score gap suggests possible generalization issues for spam detection. The test spam recall of 61.83% also shows that some spam messages are missed.

Further improvements should focus on cross-validation, validation-based model tuning, and improving spam recall while maintaining good precision.